In [1]:
!pip install datasets
!pip install transformers
!pip install tensorflow
!pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 22.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [2]:
import tensorflow as tf
from transformers import TFAutoModelForSequenceClassification, AutoConfig, AutoTokenizer
from datasets import load_dataset
import numpy as np
from sklearn.metrics import roc_auc_score
import time

class BenchmarkTrainer:
    def __init__(self, model_name="distilbert-base-uncased", max_length=32):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.max_length = max_length
        self.config = AutoConfig.from_pretrained(
            model_name,
            num_labels=2,
            finetuning_task="sequence-classification"
        )

    def prepare_dataset(self, texts, labels):
        encodings = self.tokenizer(
            texts,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='tf'
        )

        dataset = tf.data.Dataset.from_tensor_slices((
            dict(encodings),
            tf.cast(labels, tf.int32)
        ))
        return dataset.batch(16).prefetch(tf.data.AUTOTUNE)

    def prepare_benchmark_data(self):
        # Load and preprocess data
        dataset = load_dataset("Anthropic/hh-rlhf", split='train[:100%]')

        chosen_texts = dataset["chosen"]
        rejected_texts = dataset["rejected"]

        size = 20000
        texts = chosen_texts[:size // 2] + rejected_texts[:size // 2]
        labels = [1] * (size // 2) + [0] * (size // 2)

        # Create train/val split
        split_idx = int(0.7 * len(texts))
        indices = np.random.permutation(len(texts))

        train_data = self.prepare_dataset(
            [texts[i] for i in indices[:split_idx]],
            [labels[i] for i in indices[:split_idx]]
        )
        val_data = self.prepare_dataset(
            [texts[i] for i in indices[split_idx:]],
            [labels[i] for i in indices[split_idx:]]
        )

        return train_data, val_data

class RLHFBenchmark(BenchmarkTrainer):
    def train_step(self, inputs, labels, optimizer, loss_fn):
        with tf.GradientTape() as tape:
            outputs = self.model(**inputs, training=True)
            loss = loss_fn(labels, outputs.logits)

        grads = tape.gradient(loss, self.model.trainable_variables)
        optimizer.apply_gradients(zip(grads, self.model.trainable_variables))
        return loss

    def run_benchmark(self, train_data, val_data, epochs=20):
        self.model = TFAutoModelForSequenceClassification.from_pretrained(
            "distilbert-base-uncased",
            config=self.config
        )

        optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

        metrics_history = []
        start_time = time.time()

        for epoch in range(epochs):
            epoch_loss = tf.keras.metrics.Mean()

            for batch in train_data:
                inputs, labels = batch
                loss = self.train_step(inputs, labels, optimizer, loss_fn)
                epoch_loss.update_state(loss)

            # Evaluate
            all_preds = []
            all_labels = []
            for batch in val_data:
                inputs, labels = batch
                logits = self.model(**inputs, training=False).logits
                preds = tf.nn.softmax(logits)[:, 1]
                all_preds.extend(preds.numpy())
                all_labels.extend(labels.numpy())

            metrics = {
                'epoch': epoch + 1,
                'loss': float(epoch_loss.result()),
                'auc_roc': roc_auc_score(all_labels, all_preds),
                'accuracy': float(tf.reduce_mean(tf.cast(tf.round(all_preds) == all_labels, tf.float32)))
            }
            metrics_history.append(metrics)

        return {
            'training_time': time.time() - start_time,
            'metrics': metrics_history,
            'peak_memory': tf.config.experimental.get_memory_info('GPU:0')['peak'] if tf.config.list_physical_devices('GPU') else 0
        }

class DPOBenchmark(BenchmarkTrainer):
    def train_step(self, inputs, labels, optimizer):
        with tf.GradientTape() as tape:
            outputs = self.model(**inputs, training=True)
            logits = outputs.logits
            probs = tf.nn.softmax(logits, axis=-1)

            chosen_mask = labels == 1
            rejected_mask = labels == 0

            chosen_probs = tf.boolean_mask(probs[:, 1], chosen_mask)
            rejected_probs = tf.boolean_mask(probs[:, 1], rejected_mask)

            advantages = chosen_probs[:, None] - rejected_probs[None, :]
            loss = -tf.reduce_mean(tf.math.log_sigmoid(advantages))

        grads = tape.gradient(loss, self.model.trainable_variables)
        optimizer.apply_gradients(zip(grads, self.model.trainable_variables))
        return loss

    def run_benchmark(self, train_data, val_data, epochs=3):
        self.model = TFAutoModelForSequenceClassification.from_pretrained(
            "distilbert-base-uncased",
            config=self.config
        )

        optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)

        metrics_history = []
        start_time = time.time()

        for epoch in range(epochs):
            epoch_loss = tf.keras.metrics.Mean()

            for batch in train_data:
                inputs, labels = batch
                loss = self.train_step(inputs, labels, optimizer)
                epoch_loss.update_state(loss)

            # Evaluate
            all_preds = []
            all_labels = []
            for batch in val_data:
                inputs, labels = batch
                logits = self.model(**inputs, training=False).logits
                preds = tf.nn.softmax(logits)[:, 1]
                all_preds.extend(preds.numpy())
                all_labels.extend(labels.numpy())

            metrics = {
                'epoch': epoch + 1,
                'loss': float(epoch_loss.result()),
                'auc_roc': roc_auc_score(all_labels, all_preds),
                'accuracy': float(tf.reduce_mean(tf.cast(tf.round(all_preds) == all_labels, tf.float32)))
            }
            metrics_history.append(metrics)

        return {
            'training_time': time.time() - start_time,
            'metrics': metrics_history,
            'peak_memory': tf.config.experimental.get_memory_info('GPU:0')['peak'] if tf.config.list_physical_devices('GPU') else 0
        }

def run_benchmarks():
    # Set random seeds
    tf.random.set_seed(42)
    np.random.seed(42)

    # Initialize trainers
    rlhf_trainer = RLHFBenchmark()
    dpo_trainer = DPOBenchmark()

     Prepare datasets#
    train_data, val_data = rlhf_trainer.prepare_benchmark_data()

    print("Running RLHF...")
    rlhf_results = rlhf_trainer.run_benchmark(train_data, val_data)

    print("Running DPO...")
    dpo_results = dpo_trainer.run_benchmark(train_data, val_data)

    # Print results
    print("\nBenchmark Results:")
    for model, results in {"RLHF": rlhf_results, "DPO": dpo_results}.items():
        final_metrics = results['metrics'][-1]
        print(f"\n{model}:")
        print(f"Training time: {results['training_time']:.2f}s")
        print(f"Final AUC-ROC: {final_metrics['auc_roc']:.4f}")
        print(f"Final Accuracy: {final_metrics['accuracy']:.4f}")
        print(f"Peak Memory: {results['peak_memory'] / (1024**2):.2f}MB")

if __name__ == "__main__":
    run_benchmarks()


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_transform.weight', 'vocab_layer_norm.bias', 'vocab_layer_norm.weight', 'vocab_projector.bias', 'vocab_transform.bias']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight', 'classifier.bias']
You should 

Running DPO...


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_transform.weight', 'vocab_layer_norm.bias', 'vocab_layer_norm.weight', 'vocab_projector.bias', 'vocab_transform.bias']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight', 'classifier.bias']
You should 


Benchmark Results:

RLHF:
Training time: 7074.59s
Final AUC-ROC: 0.4963
Final Accuracy: 0.4940
Peak Memory: 1495.14MB

DPO:
Training time: 1114.10s
Final AUC-ROC: 0.4964
Final Accuracy: 0.5060
Peak Memory: 2316.64MB
